<a href="https://colab.research.google.com/github/Shah-Shawon/Deep-Learing-Assignments/blob/main/Assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install & Mount Drive
!pip install -q umap-learn tensorflow scikit-learn seaborn pillow
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import seaborn as sns
import tensorflow as tf # Added this import
from tensorflow.keras.applications import (
    VGG16, VGG19, ResNet50, ResNet50V2, ResNet101, InceptionV3,
    InceptionResNetV2, DenseNet121, DenseNet169, MobileNetV2
)
from tensorflow.keras.applications.imagenet_utils import decode_predictions, preprocess_input
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.models import Model
import warnings
warnings.filterwarnings('ignore')

print("✅ Setup complete! Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete! Drive mounted.


In [2]:
# Cell 2: Paths & Models (CHANGE YOUR PATHS HERE)
# তোমার Drive/MyDrive এ faces/ ও flowers/ ফোল্ডার রাখো
BASE_PATH = '/content/drive/MyDrive/'  # তোমার actual path দাও
FACES_PATH = BASE_PATH + 'face_images'       # faces/ ফোল্ডার
FLOWERS_PATH = BASE_PATH + 'flower_images'   # flowers/ ফোল্ডার

# 10 Pre-trained models
MODELS = {
    'VGG16': VGG16(weights='imagenet', include_top=True),
    'VGG19': VGG19(weights='imagenet', include_top=True),
    'ResNet50': ResNet50(weights='imagenet', include_top=True),
    'ResNet50V2': ResNet50V2(weights='imagenet', include_top=True),
    'ResNet101': ResNet101(weights='imagenet', include_top=True),
    'InceptionV3': InceptionV3(weights='imagenet', include_top=True),
    'InceptionResNetV2': InceptionResNetV2(weights='imagenet', include_top=True),
    'DenseNet121': DenseNet121(weights='imagenet', include_top=True),
    'DenseNet169': DenseNet169(weights='imagenet', include_top=True),
    'MobileNetV2': MobileNetV2(weights='imagenet', include_top=True)
}

IMG_SIZE = 224

# Define target sizes for each model
MODEL_TARGET_SIZES = {
    'VGG16': (224, 224),
    'VGG19': (224, 224),
    'ResNet50': (224, 224),
    'ResNet50V2': (224, 224),
    'ResNet101': (224, 224),
    'InceptionV3': (299, 299),
    'InceptionResNetV2': (299, 299),
    'DenseNet121': (224, 224),
    'DenseNet169': (224, 224),
    'MobileNetV2': (224, 224)
}

553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step
574710816/574710816 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step
102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
102869336/102869336 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
179648224/179648224 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
96112376/96112376 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
225209952/225209952 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
33188688/33188688 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
58541896/58541896 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [3]:
# Cell 3: Load Images Function
def load_images(folder_path, target_size=(IMG_SIZE, IMG_SIZE)):
    images = []
    filenames = []
    labels = []
    label_map = {}  # filename -> label

    print(f"Loading from {folder_path}...")
    for filename in sorted(os.listdir(folder_path)):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(folder_path, filename)
            try:
                img = keras_image.load_img(img_path, target_size=target_size)
                img_array = keras_image.img_to_array(img)
                images.append(img_array)
                filenames.append(filename)

                # Auto-labeling (customize করো)
                if 'male' in filename.lower() or 'm' in filename.lower():
                    labels.append('Male')
                elif 'female' in filename.lower() or 'f' in filename.lower():
                    labels.append('Female')
                elif 'my' in filename.lower() or 'self' in filename.lower():
                    labels.append('MyFace')
                else:
                    labels.append('Face')  # faces folder এ default

                print(f"Loaded: {filename} -> {labels[-1]}")
            except:
                print(f"Skip: {filename}")

    return np.array(images), filenames, labels


In [4]:
# Cell 4: Load Data from SUBFOLDERS (নতুন version)
import glob

def load_images_subfolders(folder_path):
    """Load images from subfolders - subfolder name = label"""
    images = []
    filenames = []
    labels = []

    print(f"🔍 Scanning {folder_path}...")

    # Find all image files in all subfolders recursively
    image_extensions = ['*.jpg', '*.jpeg', '*.png']
    all_image_paths = []

    for ext in image_extensions:
        all_image_paths.extend(glob.glob(f"{folder_path}/**/{ext}", recursive=True))

    print(f"Found {len(all_image_paths)} images in subfolders")

    for img_path in sorted(all_image_paths):
        try:
            # Extract label from subfolder name
            label = os.path.basename(os.path.dirname(img_path))
            filename = os.path.basename(img_path)

            # Load image
            img = keras_image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
            img_array = keras_image.img_to_array(img)

            images.append(img_array)
            filenames.append(filename)
            labels.append(label)

            if len(images) <= 3:  # Show first few
                print(f"  {label}/{filename}")

        except Exception as e:
            print(f"Skip {img_path}: {e}")

    print(f"✅ Loaded {len(images)} images with {len(set(labels))} unique labels")
    return np.array(images), filenames, labels

# Load YOUR Data
print("🔄 Loading FACES from subfolders...")
faces_imgs, faces_names, face_labels = load_images_subfolders(FACES_PATH)
print(f"✅ Face labels: {sorted(set(face_labels))}")

print("\n🔄 Loading FLOWERS from subfolders...")
flowers_imgs, flowers_names, flower_labels = load_images_subfolders(FLOWERS_PATH)
print(f"✅ Flower labels: {sorted(set(flower_labels))}")

# Quick preview
print(f"\n📸 Face samples: {face_labels[:5]}")
print(f"📸 Flower samples: {flower_labels[:5]}")

🔄 Loading FACES from subfolders...
🔍 Scanning /content/drive/MyDrive/face_images...
Found 384 images in subfolders
  Aziz/0048873e-c73f-48d6-be33-54b6a34b5d39.jpg
  Aziz/1623f748-634d-4188-8a26-28207ba8e100.jpg
  Aziz/17e265cb-64cc-414c-913e-b867bd4fbe4f.jpg
✅ Loaded 384 images with 6 unique labels
✅ Face labels: ['Aziz', 'Shah_mostafa_shawon', 'Sujan prodhan', 'Sumon', 'Tanjim', 'Yeasin']

🔄 Loading FLOWERS from subfolders...
🔍 Scanning /content/drive/MyDrive/flower_images...
Found 48 images in subfolders
  Flower1/6255757434886491763_121.jpg
  Flower1/6255757434886491764_121.jpg
  Flower1/6255757434886491765_121.jpg
✅ Loaded 48 images with 7 unique labels
✅ Flower labels: ['Flower1', 'Flower2', 'Flower3', 'Hibiscus', 'Little Sunflower', 'Sunflower', 'merigold']

📸 Face samples: ['Aziz', 'Aziz', 'Aziz', 'Aziz', 'Aziz']
📸 Flower samples: ['Flower1', 'Flower1', 'Flower1', 'Flower1', 'Flower1']


In [5]:
# Cell 5: Prediction Functions
def preprocess_batch(imgs, target_size=(IMG_SIZE, IMG_SIZE)):
    imgs = np.array(imgs)
    if len(imgs.shape) == 3: # If it's [H, W, C], add batch dimension
        imgs = np.expand_dims(imgs, axis=0)

    current_height, current_width = imgs.shape[1:3]
    required_height, required_width = target_size

    if current_height != required_height or current_width != required_width:
        print(f"Resizing images from ({current_height}, {current_width}) to ({required_height}, {required_width}) for current model.")
        imgs = tf.image.resize(imgs, [required_height, required_width]).numpy()

    return preprocess_input(imgs.copy())

def get_predictions(model, imgs_preprocessed):
    preds = model.predict(imgs_preprocessed, verbose=0)
    # decode_predictions returns a list of lists of tuples, e.g., [[('id1', 'name1', score1)], [('id2', 'name2', score2)]]
    # We need to extract the inner tuple for each image to get a list of tuples for top1.
    top1_predictions_for_batch = [item[0] for item in decode_predictions(preds, top=1)]
    top5_predictions_for_batch = decode_predictions(preds, top=5)
    return top1_predictions_for_batch, top5_predictions_for_batch

def get_feature_extractor(model_name):
    model = MODELS[model_name]
    # Remove classification head
    feat_model = Model(inputs=model.input, outputs=model.layers[-2].output)

In [6]:
# Cell 6: CLASSIFICATION (Top-1, Top-5)
print("\n🎯 CLASSIFICATION STARTED...")

def classify_dataset(imgs, names, labels, data_type):
    # Added check to prevent IndexError if no images are loaded
    if len(imgs) == 0:
        print(f"Warning: No images found for {data_type}. Skipping classification for this dataset.")
        return {}

    results = {}

    for model_name, model in MODELS.items():
        print(f"{data_type}: {model_name}...")
        # Get the target size for the current model
        model_target_size = MODEL_TARGET_SIZES.get(model_name, (IMG_SIZE, IMG_SIZE)) # Default to IMG_SIZE if not found

        imgs_pre = preprocess_batch(imgs, model_target_size) # Pass target_size to preprocess_batch
        top1, top5 = get_predictions(model, imgs_pre)

        results[model_name] = {
            'top1': [(names[i], top1[i][1], top1[i][2]) for i in range(len(names))],
            'top5': [[(names[i], [p[1] for p in top5[i][:5]], [p[2] for p in top5[i][:5]])] for i in range(len(names))]
        }

    # Save predictions
    with open(f'/content/{data_type}_predictions.txt', 'w') as f:
        f.write(f"{data_type.upper()} RESULTS:\n\n")
        for model_name, res in results.items():
            f.write(f"=== {model_name} ===\n")
            for i, (name, t1_name, t1_score) in enumerate(res['top1']):
                f.write(f"{name} (Label:{labels[i]}): Top1={t1_name}({t1_score:.3f})\n")
            f.write("\n")
    return results

face_results = classify_dataset(faces_imgs, faces_names, face_labels, 'faces')
flower_results = classify_dataset(flowers_imgs, flowers_names, flower_labels, 'flowers')

print("✅ Predictions saved: /content/faces_predictions.txt, /content/flowers_predictions.txt")


🎯 CLASSIFICATION STARTED...
faces: VGG16...
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
faces: VGG19...
faces: ResNet50...
faces: ResNet50V2...
faces: ResNet101...
faces: InceptionV3...
Resizing images from (224, 224) to (299, 299) for current model.
faces: InceptionResNetV2...
Resizing images from (224, 224) to (299, 299) for current model.
faces: DenseNet121...
faces: DenseNet169...
faces: MobileNetV2...
flowers: VGG16...
flowers: VGG19...
flowers: ResNet50...
flowers: ResNet50V2...


flowers: ResNet101...


flowers: InceptionV3...
Resizing images from (224, 224) to (299, 299) for current model.
flowers: InceptionResNetV2...
Resizing images from (224, 224) to (299, 299) for current model.
flowers: DenseNet121...
flowers: DenseNet169...
flowers: MobileNetV2...
✅ Predictions saved: /content/faces_predictions.txt, /content/flowers_predictions.txt


In [7]:
# Cell 7: FIXED FEATURE EXTRACTION & VISUALIZATION ✅
print("\n📊 FEATURE EXTRACTION STARTED...")

def get_feature_extractor_fixed(model_name):
    """Fixed feature extractor - proper layer indexing"""
    try:
        model = MODELS[model_name]

        # Different models have different final layer structures
        if 'VGG' in model_name:
            # VGG16/19: layer -2 is correct (before softmax)
            feat_model = Model(inputs=model.input, outputs=model.layers[-2].output)
        elif 'ResNet' in model_name or 'DenseNet' in model_name:
            # ResNet/DenseNet: GlobalAveragePooling2D before top
            for i, layer in enumerate(model.layers):
                if 'global_average_pooling' in layer.name.lower() or 'pool' in layer.name.lower():
                    feat_model = Model(inputs=model.input, outputs=model.layers[i].output)
                    print(f"    Using layer {i}: {layer.name} for {model_name}")
                    return feat_model
            # Fallback to -2
            feat_model = Model(inputs=model.input, outputs=model.layers[-2].output)
        else:
            # Inception/MobileNet fallback
            feat_model = Model(inputs=model.input, outputs=model.layers[-2].output)

        print(f"    Feature extractor ready for {model_name}")
        return feat_model

    except Exception as e:
        print(f"    ERROR {model_name}: {e}")
        return None

def reduce_dimensions(features, method='pca', n_components=2):
    if method == 'pca':
        reducer = PCA(n_components=n_components, random_state=42)
    elif method == 'tsne':
        reducer = TSNE(n_components=n_components, random_state=42, perplexity=min(10, len(features)-1))
    elif method == 'umap':
        reducer = umap.UMAP(n_components=n_components, random_state=42, n_neighbors=min(15, len(features)-1))
    return reducer.fit_transform(features)

def plot_2d(X_2d, labels, title, model_name, method, out_path):
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x=X_2d[:,0], y=X_2d[:,1], hue=labels, palette='tab10', s=120, alpha=0.8)
    plt.title(f'{title}\n{model_name} - {method.upper()}', fontsize=16, pad=20)
    plt.xlabel('Dimension 1')
    plt.ylabel('Dimension 2')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{out_path}/{title.lower()}_{model_name}_{method}.png', dpi=300, bbox_inches='tight')
    plt.close()
    return True

# Sample data for speed (first 30 images per class)
def sample_data(imgs, labels, max_per_class=30):
    unique_labels = list(set(labels))
    sampled_imgs = []
    sampled_labels = []

    for label in unique_labels:
        idx = [i for i, l in enumerate(labels) if l == label]
        n_sample = min(max_per_class, len(idx))
        sampled_imgs.extend(imgs[idx[:n_sample]])
        sampled_labels.extend([label] * n_sample)

    return np.array(sampled_imgs), sampled_labels

print("⏳ Sampling data for visualization...")
faces_sampled, face_labels_sampled = sample_data(faces_imgs, face_labels, 20)
flowers_sampled, flower_labels_sampled = sample_data(flowers_imgs, flower_labels, 15)
print(f"  Faces: {len(faces_sampled)} images, {len(set(face_labels_sampled))} classes")
print(f"  Flowers: {len(flowers_sampled)} images, {len(set(flower_labels_sampled))} classes")

# Create outputs
os.makedirs('/content/outputs_faces', exist_ok=True)
os.makedirs('/content/outputs_flowers', exist_ok=True)

all_data = [
    ('faces', faces_sampled, face_labels_sampled, '/content/outputs_faces'),
    ('flowers', flowers_sampled, flower_labels_sampled, '/content/outputs_flowers')
]

for data_type, imgs, labels, out_path in all_data:
    print(f"\n🔄 Processing {data_type}...")
    imgs_pre = preprocess_batch(imgs)

    for model_name in ['VGG16', 'ResNet50', 'DenseNet121'][:2]:  # 2 models fast
        print(f"  {model_name}...")

        feat_extractor = get_feature_extractor_fixed(model_name)
        if feat_extractor is None:
            print(f"  ❌ Skipping {model_name}")
            continue

        # Extract features
        features = feat_extractor.predict(imgs_pre[:50], verbose=0)  # First 50
        features = features.reshape(features.shape[0], -1)
        print(f"    Features shape: {features.shape}")

        # 3 methods
        for method in ['pca', 'tsne', 'umap']:
            try:
                X_2d = reduce_dimensions(features, method)
                plot_2d(X_2d, labels[:features.shape[0]], data_type.title(), model_name, method, out_path)
                print(f"    ✅ {method.upper()}.png saved")
            except Exception as e:
                print(f"    ❌ {method}: {e}")

print("\n🎉 ALL FEATURE PLOTS SAVED!")
print("📁 Check: /content/outputs_faces/ & /content/outputs_flowers/")


📊 FEATURE EXTRACTION STARTED...
⏳ Sampling data for visualization...
  Faces: 120 images, 6 classes
  Flowers: 48 images, 7 classes

🔄 Processing faces...
  VGG16...
    Feature extractor ready for VGG16
    Features shape: (50, 4096)
    ✅ PCA.png saved
    ✅ TSNE.png saved
    ✅ UMAP.png saved
  ResNet50...
    Using layer 5: pool1_pad for ResNet50
    Features shape: (50, 831744)
    ✅ PCA.png saved
    ✅ TSNE.png saved
    ✅ UMAP.png saved

🔄 Processing flowers...
  VGG16...
    Feature extractor ready for VGG16
    Features shape: (48, 4096)
    ✅ PCA.png saved
    ✅ TSNE.png saved
    ✅ UMAP.png saved
  ResNet50...
    Using layer 5: pool1_pad for ResNet50
    Features shape: (48, 831744)
    ✅ PCA.png saved
    ✅ TSNE.png saved
    ✅ UMAP.png saved

🎉 ALL FEATURE PLOTS SAVED!
📁 Check: /content/outputs_faces/ & /content/outputs_flowers/


In [8]:
# Cell 8: DOWNLOAD ALL OUTPUTS
from google.colab import files
import shutil

# Zip everything
shutil.make_archive('/content/assignment_outputs', 'zip', '/content/outputs_faces')
shutil.make_archive('/content/assignment_outputs_flowers', 'zip', '/content/outputs_flowers')

# Download
files.download('/content/faces_predictions.txt')
files.download('/content/flowers_predictions.txt')
files.download('/content/assignment_outputs.zip')
files.download('/content/assignment_outputs_flowers.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>